In [11]:
# ==========================================
# BLOCK 1: IMPORTS & ENVIRONMENT SETUP
# ==========================================
import os
import zipfile
import glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.utils.class_weight import compute_class_weight
import tensorflow as tf
from tensorflow.keras import layers, models, Input
from tensorflow.keras.utils import plot_model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
# ====================
# BLOCK 2: LOAD DATA
# ====================

ZIP_PATH = '/content/drive/MyDrive/MLF_Project/x_train.zip'
CSV_PATH = '/content/drive/MyDrive/MLF_Project/y_train.csv'
EXTRACT_DIR = '/content/MLF_Project/x_train_unzipped'

if not os.path.exists(EXTRACT_DIR):
    os.makedirs(EXTRACT_DIR)
    print("Extracting training data...")
    with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
        zip_ref.extractall(EXTRACT_DIR)

labels_df = pd.read_csv(CSV_PATH)
X_data, y_data, ids_data = [], [], []

for index, row in labels_df.iterrows():
    csv_id = int(row.iloc[0])
    label_val = int(row.iloc[1])

    real_img_id = csv_id + 1

    img_path = os.path.join(EXTRACT_DIR, f"img_{real_img_id}.png")

    if os.path.exists(img_path):
        img = Image.open(img_path).convert('L')
        img_array = np.array(img, dtype=np.float32) / 255.0
        img_array = np.expand_dims(img_array, axis=-1)

        X_data.append(img_array)
        y_data.append(label_val)
        ids_data.append(real_img_id)

X_data = np.array(X_data, dtype=np.float32)
y_data = np.array(y_data, dtype=np.int32)
ids_data = np.array(ids_data, dtype=np.int32)
print(f"Successfully loaded and correctly paired {len(X_data)} images!")

Extracting training data...
Successfully loaded and correctly paired 9227 images!


In [4]:
# ==========================================
# BLOCK 3: DATA SPLIT (THE GOLDEN 90/10 RATIO)
# ==========================================

X_train, X_val, y_train, y_val, ids_train, ids_val = train_test_split(
    X_data, y_data, ids_data, test_size=0.1, random_state=42, stratify=y_data
)

print(f"Training set ready: {X_train.shape}")
print(f"Validation set ready: {X_val.shape}")

Training set ready: (8304, 45, 51, 1)
Validation set ready: (923, 45, 51, 1)


In [5]:
# ==========================================
# BLOCK 4: THE DIVERSE ENSEMBLE FACTORY
# ==========================================

input_shape = X_train.shape[1:]

def build_diverse_model(model_version=1):
    inputs = Input(shape=input_shape)

    # mirroring
    x = layers.RandomFlip("horizontal")(inputs)

    # EXPERT 2 uses large 5x5 filters at the beginning to detect coarser shapes
    if model_version == 2:
        x = layers.Conv2D(32, (5, 5), activation='relu', padding='same')(x)
    else:
        x = layers.Conv2D(32, (3, 3), activation='relu', padding='same')(x)

    x = layers.BatchNormalization()(x)

    x = layers.Conv2D(64, (3, 3), strides=(2, 2), activation='relu', padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.2)(x)

    x = layers.Conv2D(128, (3, 3), strides=(2, 2), activation='relu', padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.3)(x)

    x = layers.Conv2D(128, (3, 3), activation='relu', padding='same')(x)
    x = layers.BatchNormalization()(x)

    x = layers.Flatten()(x)

    # EXPERT 3 has a huge brain at the end, but heavier Dropout
    if model_version == 3:
        x = layers.Dense(512, activation='relu')(x)
        x = layers.BatchNormalization()(x)
        x = layers.Dropout(0.6)(x)
    else:
        x = layers.Dense(256, activation='relu')(x)
        x = layers.BatchNormalization()(x)
        x = layers.Dropout(0.5)(x)

    outputs = layers.Dense(4, activation='softmax')(x)

    model = models.Model(inputs, outputs)
    model.compile(optimizer=Adam(learning_rate=0.0005),
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])
    return model

print("Diverse model factory is ready!")

Diverse model factory is ready!


In [12]:
# ==========================================
# BLOCK 5: TRAINING THE DIVERSE COUNCIL
# ==========================================

print("Computing class weights...")
weights = compute_class_weight(class_weight='balanced', classes=np.unique(y_train), y=y_train)
class_weights_dict = dict(enumerate(weights))

NUM_MODELS = 3
ensemble_models = []
ensemble_histories = []

for i in range(1, NUM_MODELS + 1):
    print(f"\n{'='*50}")
    print(f"TRAINING EXPERT {i} / {NUM_MODELS}")
    if i == 1: print("   (Type: Golden Standard)")
    if i == 2: print("   (Type: Macro-expert with 5x5 filters)")
    if i == 3: print("   (Type: Deep Analyst with Dense 512)")
    print(f"{'='*50}")

    model = build_diverse_model(model_version=i)

    reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=0.00001, verbose=1)
    early_stop = EarlyStopping(monitor='val_loss', patience=18, restore_best_weights=True, verbose=1)

    history = model.fit(
        X_train, y_train,
        epochs=80,
        batch_size=32,
        validation_data=(X_val, y_val),
        class_weight=class_weights_dict,
        callbacks=[reduce_lr, early_stop],
        verbose=1
    )

    ensemble_models.append(model)
    ensemble_histories.append(history)

print("\nALL 3 EXPERTS SUCCESSFULLY TRAINED AND READY TO VOTE!")

Computing class weights...

TRAINING EXPERT 1 / 3
   (Type: Golden Standard)
Epoch 1/80
260/260 ━━━━━━━━━━━━━━━━━━━━ 22s 44ms/step - accuracy: 0.6189 - loss: 0.9359 - val_accuracy: 0.3250 - val_loss: 2.2563 - learning_rate: 5.0000e-04
Epoch 2/80
260/260 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - accuracy: 0.7365 - loss: 0.5833 - val_accuracy: 0.4160 - val_loss: 1.2501 - learning_rate: 5.0000e-04
Epoch 3/80
260/260 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - accuracy: 0.7950 - loss: 0.4394 - val_accuracy: 0.7627 - val_loss: 0.5487 - learning_rate: 5.0000e-04
Epoch 4/80
260/260 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - accuracy: 0.8230 - loss: 0.3805 - val_accuracy: 0.7887 - val_loss: 0.5636 - learning_rate: 5.0000e-04
Epoch 5/80
260/260 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - accuracy: 0.8468 - loss: 0.3414 - val_accuracy: 0.7692 - val_loss: 0.5569 - learning_rate: 5.0000e-04
Epoch 6/80
260/260 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - accuracy: 0.8681 - loss: 0.2966 - val_accuracy: 0.8732 - val_loss: 0.3334 - learning_ra

In [ ]:
# =======================
# BLOCK 6: KAGGLE EXPORT
# =======================
TEST_ZIP_PATH = '/content/drive/MyDrive/MLF_Project/x_test.zip'
TEST_EXTRACT_DIR = '/content/MLF_Project/x_test_unzipped'
SUBMISSION_CSV_PATH = '/content/drive/MyDrive/MLF_Project/JDVL_FINAL.csv'

if not os.path.exists(TEST_EXTRACT_DIR):
    os.makedirs(TEST_EXTRACT_DIR)
    import zipfile
    with zipfile.ZipFile(TEST_ZIP_PATH, 'r') as zip_ref:
        zip_ref.extractall(TEST_EXTRACT_DIR)

print("Loading Kaggle images...")
test_image_paths = glob.glob(os.path.join(TEST_EXTRACT_DIR, '*.png'))

kaggle_ids = []
test_images = []

for img_path in test_image_paths:
    img_number = int(''.join(filter(str.isdigit, os.path.basename(img_path))))
    kaggle_ids.append(img_number - 1)

    img = Image.open(img_path).convert('L')
    img_array = np.array(img, dtype=np.float32) / 255.0
    img_array = np.expand_dims(img_array, axis=-1)
    test_images.append(img_array)

X_test_batch = np.array(test_images)

# TTA: Create a mirrored copy of the test data
X_test_batch_flipped = np.flip(X_test_batch, axis=2)

print(f"Starting brutal voting (3 different experts x 2 views = 6 votes for each photo)...")
summed_probs = np.zeros((len(X_test_batch), 4))

for idx, m in enumerate(ensemble_models):
    print(f"Expert {idx+1} is adding its normal and mirrored vote...")
    summed_probs += m.predict(X_test_batch, batch_size=32, verbose=0)
    summed_probs += m.predict(X_test_batch_flipped, batch_size=32, verbose=0)

pred_classes = np.argmax(summed_probs, axis=1)

predictions = [{'id': k_id, 'Target': p_class} for k_id, p_class in zip(kaggle_ids, pred_classes)]

submission_df = pd.DataFrame(predictions).sort_values(by='id')
submission_df.to_csv(SUBMISSION_CSV_PATH, index=False)

print(f"\nDONE! Here is your file: {SUBMISSION_CSV_PATH}")

In [13]:
# ==========================================
# BLOCK 7: GENERATE GITHUB PICTURES
# ==========================================

PICTURES_DIR = 'pictures'
os.makedirs(PICTURES_DIR, exist_ok=True)
print(f"Directory '{PICTURES_DIR}' is ready.")

# ---------------------------------------------------------
# 1. GENERATE DATA SAMPLES (data_samples.png)
# ---------------------------------------------------------
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
fig.suptitle('Radar Data Samples by Person Count', fontsize=16, fontweight='bold', y=1.05)

for i in range(4):
    idx = np.where(y_train == i)[0][0]
    img = X_train[idx].squeeze()

    axes[i].imshow(img, cmap='gray')
    axes[i].set_title(f'Target: {i} Persons', fontsize=14)
    axes[i].axis('off')

plt.tight_layout()
plt.savefig(os.path.join(PICTURES_DIR, 'data_samples.png'), bbox_inches='tight', dpi=150)
plt.close()
print("-> 'data_samples.png' saved successfully.")

# ---------------------------------------------------------
# 2. GENERATE CONFUSION MATRIX (confusion_matrix.png)
# ---------------------------------------------------------
val_summed_probs = np.zeros((len(X_val), 4))
for m in ensemble_models:
    val_summed_probs += m.predict(X_val, batch_size=32, verbose=0)

y_val_pred = np.argmax(val_summed_probs, axis=1)
cm = confusion_matrix(y_val, y_val_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['0 Persons', '1 Person', '2 Persons', '3 Persons'],
            yticklabels=['0 Persons', '1 Person', '2 Persons', '3 Persons'],
            cbar=False)

plt.title('Ensemble Validation Confusion Matrix', pad=20, fontsize=14, fontweight='bold')
plt.xlabel('Predicted Label', fontsize=12, labelpad=10)
plt.ylabel('True Label', fontsize=12, labelpad=10)
plt.tight_layout()
plt.savefig(os.path.join(PICTURES_DIR, 'confusion_matrix.png'), bbox_inches='tight', dpi=150)
plt.close()
print("-> 'confusion_matrix.png' saved successfully.")

# ---------------------------------------------------------
# 3. GENERATE TRAINING HISTORY (training_history.png)
# ---------------------------------------------------------
print("Generating training history graphs...")
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Training History of the Diverse Ensemble', fontsize=18, fontweight='bold', y=1.02)

for i, history in enumerate(ensemble_histories):
    # Accuracy plot (top row)
    axes[0, i].plot(history.history['accuracy'], label='Train Accuracy', color='#1f77b4', linewidth=2)
    axes[0, i].plot(history.history['val_accuracy'], label='Val Accuracy', color='#ff7f0e', linewidth=2)
    axes[0, i].set_title(f'Expert {i+1} Accuracy', fontsize=14)
    axes[0, i].set_xlabel('Epoch', fontsize=12)
    axes[0, i].set_ylabel('Accuracy', fontsize=12)
    axes[0, i].legend()
    axes[0, i].grid(True, linestyle='--', alpha=0.7)

    # Loss plot (bottom row)
    axes[1, i].plot(history.history['loss'], label='Train Loss', color='#1f77b4', linewidth=2)
    axes[1, i].plot(history.history['val_loss'], label='Val Loss', color='#ff7f0e', linewidth=2)
    axes[1, i].set_title(f'Expert {i+1} Loss', fontsize=14)
    axes[1, i].set_xlabel('Epoch', fontsize=12)
    axes[1, i].set_ylabel('Loss', fontsize=12)
    axes[1, i].legend()
    axes[1, i].grid(True, linestyle='--', alpha=0.7)

plt.tight_layout()
plt.savefig(os.path.join(PICTURES_DIR, 'training_history.png'), bbox_inches='tight', dpi=150)
plt.close()
print("-> 'training_history.png' saved successfully.")

Directory 'pictures' is ready.
-> 'data_samples.png' saved successfully.
-> 'confusion_matrix.png' saved successfully.
Generating training history graphs...
-> 'training_history.png' saved successfully.
